# Leakage-Safe Feature Analysis

The training feature pipelines are implemented in `src/the_script/features/` and `src/the_script/models/multiposition.py`. This notebook audits saved WR/RB/TE training tables and the time coverage of available inputs; it is an exploratory companion, not the source of production features.

## Feature groups in the current pipeline

- **Player role and opportunity:** prior targets, target share, receptions, snap share, depth-chart role, and role changes.
- **Team and game script:** pass/rush rates, trailing tendencies, play volume, scoring, and QB context.
- **Matchup:** prior opponent-allowed results and defensive drive rates.
- **Receiver usage quality:** target depth, end-zone usage, and red-zone target share.
- **Availability:** player and teammate injury-report status when provided by the source.
- **Experimental only:** exponentially weighted opportunity and NFL Next Gen Stats receiving features. They are excluded from default model feature lists pending stronger validation evidence.

All history features are shifted so the target game's outcomes are excluded. Next Gen week-0 season summaries are ignored; missing low-volume player-weeks stay missing rather than being treated as zero.

## Inspect feature coverage

The next cell loads the existing local feature tables and reports data availability. Rebuild them first with the README's commands if the tables do not exist or the upstream inputs have changed.

In [2]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
feature_tables = {
    position: pl.read_parquet(ROOT / "data" / "processed" / f"{position.lower()}_player_features.parquet")
    for position in ("WR", "RB", "TE")
}

coverage_rows = []
for position, frame in feature_tables.items():
    row = {"position": position, "rows": frame.height, "players": frame["player_id"].n_unique()}
    feature_candidates = {
        "targets_avg_last_5": ("targets_avg_last_5",),
        "target_share_avg_last_5": ("target_share_avg_last_5", "pbp_target_share_avg_last_5"),
        "team_trailing_play_rate_avg_last_5": ("team_trailing_play_rate_avg_last_5",),
        "red_zone_target_share_avg_last_5": ("red_zone_target_share_avg_last_5",),
        "ngs_avg_separation_avg_last_5": ("ngs_avg_separation_avg_last_5",),
        "ngs_avg_yac_above_expectation_avg_last_5": ("ngs_avg_yac_above_expectation_avg_last_5",),
    }
    for label, candidates in feature_candidates.items():
        column = next((candidate for candidate in candidates if candidate in frame.columns), None)
        row[f"{label}_coverage"] = frame[column].is_not_null().mean() if column else None
    coverage_rows.append(row)

pl.DataFrame(coverage_rows)


position,rows,players,targets_avg_last_5_coverage,target_share_avg_last_5_coverage,team_trailing_play_rate_avg_last_5_coverage,red_zone_target_share_avg_last_5_coverage,ngs_avg_separation_avg_last_5_coverage,ngs_avg_yac_above_expectation_avg_last_5_coverage
str,i64,i64,f64,f64,f64,f64,f64,f64
"""WR""",22641,681,0.969922,0.969922,0.994523,0.969922,0.693035,0.691533
"""RB""",13389,446,0.966689,0.966689,0.993652,0.966689,null,null
"""TE""",10485,346,0.967,0.967,0.99485,0.967,0.557749,0.557654
